In [ ]:
"""
=============================================================
FILE 37 — EVENT DRIVEN AGENTS
=============================================================

CONCEPTS TAUGHT
----------------
1. Event Driven Systems
2. Reactive AI
3. Event Based Routing
4. Real-Time AI Systems
5. Monitoring Agents
6. Trigger Based Workflows
7. Autonomous Event Handling
8. AI Automation
9. Streaming Events
10. Reactive Architectures

CORE IDEA
-----------
Agents react dynamically
to incoming events.

FLOW
-----
Incoming Event
   ↓
Classify Event
   ↓
Trigger Correct Agent
   ↓
Action Execution

REAL WORLD USE CASES
---------------------
- Fraud detection
- Monitoring systems
- Alerting systems
- Stock trading AI
- Enterprise automation
"""

# ============================================================
# STEP 1 — IMPORTS
# ============================================================

import os

from dotenv import load_dotenv

from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END

from IPython.display import Image, display

# ============================================================
# STEP 2 — ENV VARIABLES
# ============================================================

load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

# ============================================================
# STEP 3 — LLM
# ============================================================

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

# ============================================================
# STEP 4 — STATE
# ============================================================

class State(TypedDict):
    event: str
    event_type: str
    action_taken: str

# ============================================================
# STEP 5 — EVENT CLASSIFIER
# ============================================================

def classify_event(state: State):

    print("\nClassifying event...\n")

    response = llm.invoke(
        f"""
        Classify this event into:
        - security
        - billing
        - infrastructure

        EVENT:
        {state['event']}
        """
    )

    classification = response.content.lower()

    if "security" in classification:
        return {"event_type": "security"}

    elif "billing" in classification:
        return {"event_type": "billing"}

    else:
        return {"event_type": "infrastructure"}

# ============================================================
# STEP 6 — ROUTER
# ============================================================

def route_event(state: State):

    return state["event_type"]

# ============================================================
# STEP 7 — EVENT HANDLERS
# ============================================================

def security_agent(state: State):

    return {
        "action_taken":
        "Security team alerted immediately."
    }

def billing_agent(state: State):

    return {
        "action_taken":
        "Billing ticket generated."
    }

def infrastructure_agent(state: State):

    return {
        "action_taken":
        "Infrastructure monitoring triggered."
    }

# ============================================================
# STEP 8 — BUILD GRAPH
# ============================================================

builder = StateGraph(State)

builder.add_node("classify_event", classify_event)

builder.add_node("security_agent", security_agent)
builder.add_node("billing_agent", billing_agent)
builder.add_node("infrastructure_agent", infrastructure_agent)

# ============================================================
# STEP 9 — DEFINE EDGES
# ============================================================

builder.add_edge(
    START,
    "classify_event"
)

builder.add_conditional_edges(
    "classify_event",
    route_event,
    {
        "security": "security_agent",
        "billing": "billing_agent",
        "infrastructure": "infrastructure_agent"
    }
)

builder.add_edge("security_agent", END)
builder.add_edge("billing_agent", END)
builder.add_edge("infrastructure_agent", END)

# ============================================================
# STEP 10 — COMPILE
# ============================================================

graph = builder.compile()

display(
    Image(
        graph.get_graph().draw_mermaid_png()
    )
)

# ============================================================
# STEP 11 — RUN WORKFLOW
# ============================================================

result = graph.invoke(
    {
        "event":
        """
        Multiple failed login attempts detected
        from foreign IP addresses.
        """
    }
)

# ============================================================
# STEP 12 — PRINT RESULT
# ============================================================

print("\nEVENT RESPONSE\n")
print("=" * 60)
print(result["action_taken"])